# This project is designed as a quick introduction to the basics of LandGraph.
# It serves as a practice notebook to review or get started with LandGraph concepts.
# The notebook ends with a simple chatbot example implemented using Gradio.


In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import random

In [ ]:
load_dotenv(override=True)

In [ ]:
nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Zombies", "Rainbows", "Eels", "Pickles", "Muffins"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "moody", "sparkly", "untrustworthy", "sarcastic", "squishy", "haunted"]

In [ ]:
#1 Create a State
class State(BaseModel):
    messages: Annotated[list, add_messages]

In [ ]:
#2 We start the graph builder
graph_builder = StateGraph(State)

In [ ]:
#3. We create the first node
def our_first_node(old_state: State) -> State:
    reply = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    messages = [{'role':'assistant', 'content':reply}]
    new_state = State(messages=messages)
    return new_state

#We add the node to de graph
graph_builder.add_node("first node", our_first_node)

In [ ]:
#4. We create de edges (secuential structure)
graph_builder.add_edge(START, "first node")
graph_builder.add_edge("first node", END)

In [ ]:
#5 execute the graph builder
graph = graph_builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
#Extra: Building a openai chatbot with landgrpah architecture
def chat(user_input: str, history):
    messages =[{'role':'user', 'content': user_input}]
    state = State(messages=messages)
    result = graph.invoke(state)
    print(result)
    return result["messages"][-1].content

gr.ChatInterface(chat, type="messages").launch()
